# GPQA-Diamond on GLM-5.2 and Anthropic Opus 4.8 (Eval Protocol)

Multiple-choice GPQA-Diamond (~198 questions, PhD-level biology/physics/chemistry). Runs **both** models head-to-head in one pass, grades by the **letter** the model picks (A/B/C/D), and prints accuracy side by side.

GPQA is single-turn text-only: no tools, no MCP, no docker, no judge. Lightest possible frontier benchmark.

**Prereqs**
- Jupyter kernel = conda `cookbook` env.
- `FIREWORKS_API_KEY` and `ANTHROPIC_API_KEY` in `training/.env`.
- GPQA is **gated** on HuggingFace (`Idavidrein/gpqa`). Go to https://huggingface.co/datasets/Idavidrein/gpqa, log in, accept the terms, then either `huggingface-cli login` or set `HF_TOKEN` in `training/.env`. Run the install cell once if imports fail.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q -e "../../.[eval]"

In [2]:
# --- edit these ---
FIREWORKS_MODEL = "fireworks_ai/accounts/fireworks/routers/glm-5p2-fast"
ORNITH_MODEL = "fireworks_ai/accounts/fireworks/models/ornith-1p0-9b"
ANTHROPIC_MODEL = "anthropic/claude-opus-4-8"                        # Claude Opus 4.8

MODELS = {
    "Ornith 9B (Fireworks)": ORNITH_MODEL,
    # "Opus 4.8 (Anthropic)": ANTHROPIC_MODEL,
    # "GLM-5.2 (Fireworks)": FIREWORKS_MODEL,
}

MAX_ROWS = 10        # smoke test: set to something like 5 - 10. Full GPQA-Diamond = 198. Set None for full run.
TEMPERATURE = 0.2
MAX_TOKENS = None     # GPQA questions are long; give reasoning room.
CONCURRENCY = 2        # per-model concurrency
REQUEST_TIMEOUT = 1800
DATASET_NAME = "Idavidrein/gpqa"
DATASET_CONFIG = "gpqa_diamond"

# "tool" -> model calls submit_answer(letter=...) and we grade from the tool call (unambiguous).
# "freetext" -> model writes "Answer: X" and we regex it (legacy, can misparse).
GRADING_MODE = "tool"

In [3]:
import asyncio
import os
import random
import re
from pathlib import Path

import litellm
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm.auto import tqdm
from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import SingleTurnRolloutProcessor

from eval_protocol.pytest.types import RolloutProcessorConfig

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")

if not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")
if not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError(f"Set ANTHROPIC_API_KEY in {training_dir / '.env'}")
# GPQA is gated. Accept any of HF_TOKEN / HUGGINGFACE_API_KEY / HUGGING_FACE_HUB_TOKEN,
# then normalize to HF_TOKEN so `datasets.load_dataset` picks it up.
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_API_KEY") or os.getenv("HUGGING_FACE_HUB_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "GPQA is gated. Set HUGGINGFACE_API_KEY (or HF_TOKEN) in training/.env after accepting "
        "terms at https://huggingface.co/datasets/Idavidrein/gpqa"
    )
os.environ["HF_TOKEN"] = hf_token

# Anthropic rejects `temperature` for Opus 4.8 reasoning models and complains about
# unknown params; Fireworks is fine with temperature. drop_params=True lets LiteLLM
# strip per-provider so we can share one code path.
litellm.drop_params = True

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TaskDefinitionModel(BaseModel):


In [4]:
import json

# Official GPQA query template (used by the dataset authors for MC evaluation).
GPQA_QUERY_TEMPLATE_FREETEXT = (
    "Answer the following multiple choice question. The last line of your response "
    "should be of the following format: 'Answer: $LETTER' (without quotes) where "
    "LETTER is one of ABCD. Think step by step before answering.\n\n"
    "{Question}\n\n"
    "A) {A}\n"
    "B) {B}\n"
    "C) {C}\n"
    "D) {D}\n"
)

# Tool-mode template: the model reasons in prose, then commits via the submit_answer tool.
GPQA_QUERY_TEMPLATE_TOOL = (
    "Answer the following multiple choice question. Think step by step, then call the "
    "`submit_answer` tool with your chosen letter (one of A, B, C, D) as the `letter` argument. "
    "Do NOT write the answer as text; use the tool.\n\n"
    "{Question}\n\n"
    "A) {A}\n"
    "B) {B}\n"
    "C) {C}\n"
    "D) {D}\n"
)

# OpenAI-style tool schema (eval-protocol/litellm translates this for Anthropic automatically).
SUBMIT_ANSWER_TOOL = {
    "type": "function",
    "function": {
        "name": "submit_answer",
        "description": "Submit your final answer to the multiple-choice question.",
        "parameters": {
            "type": "object",
            "properties": {
                "letter": {
                    "type": "string",
                    "enum": ["A", "B", "C", "D"],
                    "description": "The chosen answer letter.",
                },
            },
            "required": ["letter"],
            "additionalProperties": False,
        },
    },
}

def query_template() -> str:
    return GPQA_QUERY_TEMPLATE_TOOL if GRADING_MODE == "tool" else GPQA_QUERY_TEMPLATE_FREETEXT

In [5]:
def build_rows(max_rows: int | None) -> list[EvaluationRow]:
    """Load GPQA-Diamond, shuffle the 4 options deterministically per row, expose A/B/C/D."""
    ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
    rows: list[EvaluationRow] = []
    tmpl = query_template()
    for i, ex in enumerate(list(ds)[: (max_rows or len(ds))]):
        correct = ex["Correct Answer"]
        choices = [ex["Incorrect Answer 1"], ex["Incorrect Answer 2"], ex["Incorrect Answer 3"]]
        # Deterministic per-row shuffle so answer position doesn't leak.
        rng = random.Random(i)
        gold_index = rng.randint(0, 3)
        choices.insert(gold_index, correct)
        letters = "ABCD"
        prompt = tmpl.format(
            Question=ex["Question"],
            A=choices[0], B=choices[1], C=choices[2], D=choices[3],
        )
        row = EvaluationRow(
            messages=[{"role": "user", "content": prompt}],
            ground_truth=letters[gold_index],
        )
        row.input_metadata.row_id = f"gpqa-{i}"
        # Stash domain metadata for the breakdown cell (dataset_info is the documented slot).
        row.input_metadata.dataset_info = {
            "domain": ex.get("High-Level Domain") or "unknown",
            "subdomain": ex.get("Subdomain") or "unknown",
        }
        if GRADING_MODE == "tool":
            row.tools = [SUBMIT_ANSWER_TOOL]
        rows.append(row)
    return rows

rows = build_rows(MAX_ROWS)
print(f"Loaded {len(rows)} GPQA-Diamond rows ({'smoke test' if MAX_ROWS else 'full run'}), grading={GRADING_MODE}")

Loaded 10 GPQA-Diamond rows (smoke test), grading=tool


In [6]:
_ANSWER_RE = re.compile(r"Answer:\s*([A-D])", re.IGNORECASE)
_STANDALONE_RE = re.compile(r"\b([A-D])\b")


def extract_letter(text: str) -> str | None:
    """Prefer 'Answer: X'; fall back to the last standalone A/B/C/D token."""
    if not text:
        return None
    m = list(_ANSWER_RE.finditer(text))
    if m:
        return m[-1].group(1).upper()
    standalone = list(_STANDALONE_RE.finditer(text))
    return standalone[-1].group(1).upper() if standalone else None


def _assistant_tool_calls(row: EvaluationRow):
    """Yield (name, arguments_str) for every tool call on the final assistant message."""
    for msg in row.messages:
        if getattr(msg, "role", None) != "assistant":
            continue
        for tc in (getattr(msg, "tool_calls", None) or []):
            fn = getattr(tc, "function", None)
            name = getattr(fn, "name", None) if fn else None
            args = getattr(fn, "arguments", None) if fn else None
            if name:
                yield name, args


def model_response(row: EvaluationRow) -> str:
    return str(row.messages[-1].content) if row.messages else ""


def user_question(row: EvaluationRow) -> str:
    for m in row.messages:
        if getattr(m, "role", None) == "user":
            return str(m.content)
    return ""


def grade_letter(row: EvaluationRow) -> EvaluateResult:
    """Freetext grader: pull 'Answer: X' (or last standalone letter) from the response text."""
    pred = extract_letter(model_response(row))
    gt = str(row.ground_truth).strip().upper()
    if pred is None:
        return EvaluateResult(score=0.0, reason="no letter parsed")
    ok = pred == gt
    return EvaluateResult(score=1.0 if ok else 0.0, reason=f"pred={pred} gt={gt}")


def grade_toolcall(row: EvaluationRow) -> EvaluateResult:
    """Tool grader: read the letter from a submit_answer(...) tool call. Unambiguous."""
    gt = str(row.ground_truth).strip().upper()
    preds = []
    for name, raw_args in _assistant_tool_calls(row):
        if name != "submit_answer":
            continue
        try:
            args = json.loads(raw_args) if isinstance(raw_args, str) else (raw_args or {})
        except Exception:
            args = {}
        letter = (args.get("letter") or "").strip().upper()
        if letter in "ABCD":
            preds.append(letter)
    if not preds:
        return EvaluateResult(score=0.0, reason="no submit_answer tool call")
    pred = preds[-1]
    ok = pred == gt
    return EvaluateResult(
        score=1.0 if ok else 0.0,
        reason=f"pred={pred} gt={gt}" + (" (multi calls, used last)" if len(preds) > 1 else ""),
    )


def grade(row: EvaluationRow) -> EvaluateResult:
    if GRADING_MODE == "tool":
        return grade_toolcall(row)
    return grade_letter(row)


def extract_pred(row: EvaluationRow) -> str | None:
    """The letter the model committed to, via whatever channel the active grading mode uses.
    Used by the inspection DataFrame so `pred` matches what the grader actually scored."""
    if GRADING_MODE == "tool":
        preds = []
        for name, raw_args in _assistant_tool_calls(row):
            if name != "submit_answer":
                continue
            try:
                args = json.loads(raw_args) if isinstance(raw_args, str) else (raw_args or {})
            except Exception:
                args = {}
            letter = (args.get("letter") or "").strip().upper()
            if letter in "ABCD":
                preds.append(letter)
        return preds[-1] if preds else None
    return extract_letter(model_response(row))

In [7]:
async def run_model(rows: list[EvaluationRow], model: str, label: str) -> list[EvaluationRow]:
    """Run one model over a fresh copy of the rows and grade by letter."""
    # Fresh copy so each model gets its own assistant turns / evaluation_result.
    work = [r.model_copy(deep=True) for r in rows]
    for i, r in enumerate(work):
        r.input_metadata.row_id = f"{model}-{i}"

    # Opus 4.8 is a reasoning model and rejects `temperature`. Fireworks/GLM accepts it.
    # drop_params=True strips *unknown* params, but temperature is "known" to Anthropic's
    # schema and still gets sent -> 400. So omit it for anthropic/* explicitly.
    completion_params = {
        "model": model,
        "max_tokens": MAX_TOKENS,
        "timeout": REQUEST_TIMEOUT,
    }
    if not model.startswith("anthropic/"):
        completion_params["temperature"] = TEMPERATURE

    processor = SingleTurnRolloutProcessor(drop_trailing_assistant_messages=True)
    config = RolloutProcessorConfig(
        completion_params=completion_params,
        mcp_config_path="",
        semaphore=asyncio.Semaphore(CONCURRENCY),
    )
    async def _safe(run_coro, idx):
        """Run one rollout; on failure return a zero-score row with the error in `reason`."""
        try:
            return await run_coro
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
            row = work[idx]
            row.evaluation_result = EvaluateResult(score=0.0, reason=f"error: {err}")
            # Make sure a grading pass later doesn't overwrite this.
            row.messages = [*row.messages, {"role": "assistant", "content": f"<error>{err}</error>"}]
            return row

    # processor returns a list of coroutines; wrap each so a single timeout/exception
    # becomes a zero-score row instead of killing the whole gather.
    coros = list(processor(work, config))
    completed = await tqdm.gather(
        *[_safe(c, i) for i, c in enumerate(coros)],
        desc=label, total=len(work),
    )
    for row in completed:
        if row.evaluation_result is None or not str(row.evaluation_result.reason).startswith("error:"):
            row.evaluation_result = grade(row)
    return completed

In [8]:
all_results: dict[str, list[EvaluationRow]] = {}
for label, model in MODELS.items():
    print(f"\n--- Running {label} ({model}) over {len(rows)} rows ---")
    all_results[label] = await run_model(rows, model, label)
    res = all_results[label]
    scores = [r.evaluation_result.score for r in res if r.evaluation_result]
    acc = sum(scores) / len(scores) if scores else 0.0
    errors = [r for r in res if r.evaluation_result and str(r.evaluation_result.reason).startswith("error:")]
    print(f"  {label}: {acc:.1%} ({int(sum(scores))}/{len(scores)})"
          + (f"  [{len(errors)} errored]" if errors else ""))


--- Running Ornith 9B (Fireworks) (fireworks_ai/accounts/fireworks/models/ornith-1p0-9b) over 10 rows ---


Ornith 9B (Fireworks): 100%|██████████| 10/10 [00:00<00:00, 60.69it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.I

In [9]:
print("=" * 72)
print(f"{'Model':<25} {'Accuracy':>10} {'Correct':>9} {'Total':>7} {'Errored':>8}")
print("-" * 72)
for label, res in all_results.items():
    scores = [r.evaluation_result.score for r in res if r.evaluation_result]
    acc = sum(scores) / len(scores) if scores else 0.0
    errors = sum(1 for r in res if r.evaluation_result and str(r.evaluation_result.reason).startswith("error:"))
    print(f"{label:<25} {acc:>9.1%} {int(sum(scores)):>9} {len(scores):>7} {errors:>8}")
print("=" * 72)
print("Accuracy is over all rows (errored rows count as 0).")
print("For accuracy over completed rows only, divide Correct by (Total - Errored).")

Model                       Accuracy   Correct   Total  Errored
------------------------------------------------------------------------
Ornith 9B (Fireworks)          0.0%         0      10       10
Accuracy is over all rows (errored rows count as 0).
For accuracy over completed rows only, divide Correct by (Total - Errored).


In [10]:
# Diagnose errored rows: show the actual error litellm printed (which gets scrolled off
# by the "Give Feedback" spam). Pulls from in-memory all_results, so run right after the run cell.
for label, res in all_results.items():
    errs = [(i, r) for i, r in enumerate(res)
            if r.evaluation_result and str(r.evaluation_result.reason).startswith("error:")]
    if not errs:
        continue
    print(f"\n=== {label}: {len(errs)} errored rows ===")
    for i, r in errs[:5]:
        print(f"  row {i}: {r.evaluation_result.reason}")
    if len(errs) > 5:
        print(f"  ... and {len(errs) - 5} more (all same error type in observed runs)")


=== Ornith 9B (Fireworks): 10 errored rows ===
  row 0: error: NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"message":"Model not found, inaccessible, and/or
  row 1: error: NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"message":"Model not found, inaccessible, and/or
  row 2: error: NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"message":"Model not found, inaccessible, and/or
  row 3: error: NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"message":"Model not found, inaccessible, and/or
  row 4: error: NotFoundError: litellm.NotFoundError: NotFoundError: Fireworks_aiException - {"error":{"message":"Model not found, inaccessible, and/or
  ... and 5 more (all same error type in observed runs)


In [11]:
# Wrong-row inspector: for each model, show the rows it got wrong (excluding errors).
# In tool mode, also flag "grader-error suspects" — the gold letter appears in the response
# text but the parsed tool call differed (rare; usually means model waffled then mis-committed).
from collections import Counter

for label, res in all_results.items():
    wrong = [(i, r) for i, r in enumerate(res)
             if r.evaluation_result and r.evaluation_result.score == 0.0
             and not str(r.evaluation_result.reason).startswith("error:")]
    if not wrong:
        print(f"\n=== {label}: no wrong (non-error) rows ===")
        continue
    print(f"\n=== {label}: {len(wrong)} wrong (non-error) rows ===")
    for i, r in wrong[:10]:
        reason = r.evaluation_result.reason
        gt = str(r.ground_truth)
        gold_in_text = gt in (model_response(r) or "")
        suspect = "  <- gold letter appears in response text (grader-suspect)" if gold_in_text else ""
        print(f"  row {i}: {reason}{suspect}")
    if len(wrong) > 10:
        print(f"  ... and {len(wrong) - 10} more")


=== Ornith 9B (Fireworks): no wrong (non-error) rows ===


In [12]:
# Domain breakdown: accuracy per High-Level Domain. Helps tell whether a low score is
# sample skew (e.g. first 50 rows are all chemistry) vs a real model gap.
from collections import defaultdict

for label, res in all_results.items():
    by_domain = defaultdict(lambda: [0, 0])  # domain -> [correct, total]
    for r in res:
        info = getattr(r.input_metadata, "dataset_info", None) or {}
        domain = info.get("domain", "unknown") or "unknown"
        ok = r.evaluation_result.score == 1.0 if r.evaluation_result else False
        by_domain[domain][1] += 1
        if ok:
            by_domain[domain][0] += 1
    print(f"\n=== {label} by domain ===")
    for domain in sorted(by_domain, key=lambda d: -by_domain[d][1]):
        c, t = by_domain[domain]
        print(f"  {domain:<28} {c:>3}/{t:<3}  {c/t:>6.1%}")


=== Ornith 9B (Fireworks) by domain ===
  unknown                        0/10     0.0%


In [13]:
import pandas as pd

records = []
for label, res in all_results.items():
    for r in res:
        records.append({
            "model": label,
            "row_id": r.input_metadata.row_id,
            "pred": extract_pred(r),
            "gt": str(r.ground_truth),
            "score": r.evaluation_result.score if r.evaluation_result else None,
            "response_len": len(model_response(r)),
            "question_preview": user_question(r)[:120].replace("\n", " "),
        })
df = pd.DataFrame(records)
df.head(10)

AttributeError: 'dict' object has no attribute 'content'

In [ ]:
# Show the first row's full prompt + each model's response for sanity-checking.
sample = rows[0]
print("=== Sample prompt (row 0) ===")
print(user_question(sample))
print("Ground truth:", sample.ground_truth)
for label, res in all_results.items():
    print(f"\n=== {label} response (row 0) ===")
    print(model_response(res[0])[:2000])

## Notes / knobs

- **Grading mode** (`GRADING_MODE` in the config cell):
  - `"tool"` (default) — the model commits via a `submit_answer(letter=...)` tool call; grading reads the letter out of the tool call args. **Unambiguous**, no regex. This is the cleanest apples-to-apples comparison between GLM and Opus. Note: not strictly comparable to Z.ai's free-text GPQA number (they use `Answer: $LETTER`), but it eliminates grader-error noise.
  - `"freetext"` — the original `Answer: $LETTER` template + regex grader (legacy). Can misparse a stray letter from reasoning; use only to reproduce the official-style number.
- **Reasoning effort**: GLM-5.2 defaults to `max` reasoning on Fireworks; Opus 4.8 uses adaptive thinking by default via LiteLLM. Both are left at provider defaults for fairness (no `reasoning_effort`/`effort` override). To force a lower tier add `"reasoning_effort": "high"` for GLM in `completion_params`, or `"thinking": {"type": "adaptive"}` for Opus via LiteLLM's anthropic extra.
- **Full run cost**: 198 questions × 2 models, each up to 16k output tokens. Expect ~10–30 min wall time at `CONCURRENCY=8` and a few dollars of API spend.
- **GPQA is gated** — if you hit `GatedRepoError`, accept the terms at https://huggingface.co/datasets/Idavidrein/gpqa and set `HF_TOKEN`.